# Data Analysis of the filtered Measurements dataset

---

In [ ]:
import duckdb

from src.data_preprocessing.config import INPUT_FILE_MEAS_SL_enc_f

In [ ]:
con = duckdb.connect()

In [ ]:
book_state_counts = con.execute(f"""
    SELECT book_state, COUNT(*) AS count
    FROM '{INPUT_FILE_MEAS_SL_enc_f}'
    GROUP BY book_state
    ORDER BY count DESC
""").fetchdf()

print(book_state_counts)

In [ ]:
# --- NULL % for all columns ---
print("Null percentage per column:")

# Get list of all column names using DESCRIBE
columns = con.execute(f"DESCRIBE SELECT * FROM '{INPUT_FILE_MEAS_SL_enc_f}'").fetchdf()['column_name'].tolist()

results = []
total_rows = con.execute(f"SELECT COUNT(*) FROM '{INPUT_FILE_MEAS_SL_enc_f}'").fetchone()[0]

for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM '{INPUT_FILE_MEAS_SL_enc_f}'").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    results.append((col, null_percentage))

# Sort and print
results.sort(key=lambda x: x[1], reverse=True)
for col, perc in results:
    print(f"{col}: {perc:.2f}% nulls")

In [ ]:
# --- Cardinality and sparsity for measurement_name_encoded & measurement_unit_encoded ---
columns_to_check = ['measurement_name_encoded', 'measurement_unit_encoded']

for col in columns_to_check:
    result = con.execute(f"""
        SELECT
            COUNT(DISTINCT {col}) AS unique_values,
            ROUND(100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_percentage
        FROM '{INPUT_FILE_MEAS_SL_enc_f}';
    """).fetchdf()
    print(f"\n=== 📌 {col} ===")
    print(result)

    # Top 10 frequent encoded values
    top_vals = con.execute(f"""
        SELECT {col}, COUNT(*) AS freq
        FROM '{INPUT_FILE_MEAS_SL_enc_f}'
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(f"\nTop 10 frequent values for {col}:")
    print(top_vals)

con.close()

In [ ]:
# === Print schema (column names + types) ===
print("Schema of the dataset:")
schema = con.execute(f"DESCRIBE SELECT * FROM '{INPUT_FILE_MEAS_SL_enc_f}'").fetchdf()
print(schema)

# === Row count ===
row_count = con.execute(f"SELECT COUNT(*) FROM '{INPUT_FILE_MEAS_SL_enc_f}'").fetchone()[0]
print(f"\nNumber of rows: {row_count}")

# === Summary statistics of numeric columns ===
print("\nSummary statistics (numeric columns):")
summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    MIN(measure_value) AS min_measure_value,
    MAX(measure_value) AS max_measure_value,
    AVG(measure_value) AS avg_measure_value,
    MIN(is_within_limits) AS min_within_limits,
    MAX(is_within_limits) AS max_within_limits,
    AVG(is_within_limits) AS avg_within_limits
FROM '{INPUT_FILE_MEAS_SL_enc_f}';
""").fetchdf()
print(summary)

# === Distinct value counts for encoded categorical columns (optional) ===
print("\nDistinct count of measurement_name_encoded and measurement_unit_encoded:")
cat_cardinality = con.execute(f"""
SELECT
    COUNT(DISTINCT measurement_name_encoded) AS n_measurement_names,
    COUNT(DISTINCT measurement_unit_encoded) AS n_measurement_units
FROM '{INPUT_FILE_MEAS_SL_enc_f}';
""").fetchdf()
print(cat_cardinality)

In [ ]:
# === Descriptive Statistics for Numeric Columns ===
print("Full descriptive statistics for numeric columns:")
desc_stats = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    MIN(measure_value) AS min_measure_value,
    MAX(measure_value) AS max_measure_value,
    AVG(measure_value) AS avg_measure_value,
    MEDIAN(measure_value) AS median_measure_value,
    STDDEV_POP(measure_value) AS stddev_measure_value,

    MIN(is_within_limits) AS min_within_limits,
    MAX(is_within_limits) AS max_within_limits,
    AVG(is_within_limits) AS avg_within_limits,
    STDDEV_POP(is_within_limits) AS stddev_within_limits,

    MIN(measurement_name_encoded) AS min_name_encoded,
    MAX(measurement_name_encoded) AS max_name_encoded,
    AVG(measurement_name_encoded) AS avg_name_encoded,

    MIN(measurement_unit_encoded) AS min_unit_encoded,
    MAX(measurement_unit_encoded) AS max_unit_encoded,
    AVG(measurement_unit_encoded) AS avg_unit_encoded
FROM '{INPUT_FILE_MEAS_SL_enc_f}';
""").fetchdf()
print(desc_stats)

# === Full Pairwise Correlations (all numeric columns with each other) ===
numeric_cols = [
    'measure_value', 'measurement_name_encoded',
    'measurement_unit_encoded', 'is_within_limits'
]

print("\nPairwise Pearson correlations:")
for i, col1 in enumerate(numeric_cols):
    for col2 in numeric_cols[i + 1:]:
        corr = con.execute(f"""
            SELECT corr({col1}, {col2}) AS correlation
            FROM '{INPUT_FILE_MEAS_SL_enc_f}';
        """).fetchone()[0]
        print(f"Correlation({col1}, {col2}) = {corr:.4f}")

In [ ]:
con.close()